<a href="https://colab.research.google.com/github/lorenzo-marandon/projet-semestriel-discord/blob/develop/1-Recherche%20d'information%20classique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 1. - Recherche d'Information classique

Dans cette partie, nous allons mettre en œuvre des principes et modèles classiques de Recherche d'Information. Le jeu de données est une collection de livres au format texte (.txt) de Henry Rider Haggard. Jetez un œil à ces documents dans le dossier _data_.

En sortie de ce module, vous serez capable de :

- Construire un index inversé ;
- Effectuer des requêtes simples selon le modèle booléen :
- Calculer la pondération des termes selon la méthode TF-IDF ;
- Mettre en œuvre le modèle vectoriel de recherche d'information et y appliquer des requêtes.

### Import des bibliothèques logicielles et configuration

Les lignes suivantes permettent d'instancier un objet de la classe `IRSystem` représentant notre moteur de recherche et de charger les données en RAM.

In [3]:
import sys

# Vérifie si le code est exécuté sur Google Colab
if 'google.colab' in sys.modules:
    # Commandes à exécuter uniquement sur Google Colab
    !git clone https://github.com/vincentmartin/tp-information-retrieval-with-llm-student-version.git
    %cd tp-information-retrieval-with-llm-student-version
else:
    # Commandes à exécuter si ce n'est pas sur Google Colab
    print("Pas sur Google Colab, ces commandes ne seront pas exécutées.")

Cloning into 'tp-information-retrieval-with-llm-student-version'...
remote: Enumerating objects: 2262, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 2262 (delta 30), reused 37 (delta 19), pack-reused 2212 (from 1)
Receiving objects: 100% (2262/2262), 13.92 MiB | 23.00 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/tp-information-retrieval-with-llm-student-version/tp-information-retrieval-with-llm-student-version


#### Chargement des données

Les lignes ci-dessous permettent de charger les données qui sont un ensemble de 60 livres au format texte (.txt) d'[Henry Rider Haggard](https://fr.wikipedia.org/wiki/Henry_Rider_Haggard).


In [4]:
from classic_ir.IRSystem import *

# !rm -rf ./data/RiderHaggard/stemmed
ir_system = IRSystem()
ir_system.read_data('./data/RiderHaggard') # chargement des données et prétraitement des documents (stemming).

Reading in documents...
Stemming Documents...
Finished 1724.txt
    Doc 1 of 60: Finished
Jess 5898.txt
    Doc 2 of 60: Jess
Dawn 10892.txt
    Doc 3 of 60: Dawn
The Tale of Three Lions 2729.txt
    Doc 4 of 60: The Tale of Three Lions
Hunter Quatermain's Story 2728.txt
    Doc 5 of 60: Hunter Quatermain's Story
Stella Fregelius 6051.txt
    Doc 6 of 60: Stella Fregelius
Fair Margaret 9780.txt
    Doc 7 of 60: Fair Margaret
Moon of Israel 2856.txt
    Doc 8 of 60: Moon of Israel
Wisdom's Daughter (1923) 0200181.txt
    Doc 9 of 60: Wisdom's Daughter (1923)
Eric Brighteyes 2721.txt
    Doc 10 of 60: Eric Brighteyes
The Ivory Child 2841.txt
    Doc 11 of 60: The Ivory Child
Queen of the Dawn (1925) 0200381.txt
    Doc 12 of 60: Queen of the Dawn (1925)
Child of Storm 1711.txt
    Doc 13 of 60: Child of Storm
Montezuma's Daughter 1848.txt
    Doc 14 of 60: Montezuma's Daughter
The Wanderer's Necklace 3097.txt
    Doc 15 of 60: The Wanderer's Necklace
The Brethren 2762.txt
    Doc 16 of 6

In [5]:
print("Nombre de documents :", len(ir_system.docs))
print("Taille du vocabulaire :", len(ir_system.vocab))
print("Premier document (extrait) :", ir_system.docs[0][:20])
print("Premiers mots du vocabulaire :", list(ir_system.vocab)[:10])

Nombre de documents : 60
Taille du vocabulaire : 46735
Premier document (extrait) : ['titl', 'a', 'winter', 'pilgrimag', 'author', 'h', 'rider', 'haggard', 'first', 'publish', '1901', 'be', 'an', 'account', 'of', 'travel', 'through', 'palestin', 'itali', 'and']
Premiers mots du vocabulaire : ['seablu', 'burgomasterelect', '208217', 'quailedit', 'freak', 'halfpromis', 'terrificdoubtless', 'destroyedcrush', 'staysout', 'scold']


### Exercice 1. - Construction de l'index inversé

Ce premier exercice a pour objectif de construire l'index inversé non positionnel. L'attribut `self.inverted_index` est un tableau associatif qui associe une liste d'entiers (docIDs) à un mot (word).

Documentation ici https://docs.python.org/3/library/collections.html#collections.defaultdict.

Exercice : modifier la fonction `index` pour calculer l'index inversé.

**Indication** : profitez du parcours des documents pour remplir aussi `self.tf`, qui associe à chaque docID un `Counter` donnant le nombre d'occurrences de chaque mot dans ce document. Vous en aurez besoin à l'exercice 3 pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [6]:
# Exercice 1. Indexation

def index(self):
    """
    Construit l'index inversé et calcule la fréquence des termes
    dans chaque document.

    L'index inversé associe chaque terme à la liste des identifiants
    des documents dans lesquels il apparaît.

    La fréquence TF correspond au nombre d'occurrences d'un terme
    dans un document.
    """
    print("Indexing...")

    self.tf = defaultdict(Counter)
    inverted_index = defaultdict(list)

    # Initialise une liste vide pour chaque terme du vocabulaire.
    for word in self.vocab:
        inverted_index[word] = []

    # Parcourt chaque document du corpus.
    for doc_id, doc in enumerate(self.docs):

        # Calcule le nombre d'occurrences de chaque terme.
        self.tf[doc_id] = Counter(doc)

        # Ajoute l'identifiant du document pour chaque terme distinct.
        for word in self.tf[doc_id]:
            inverted_index[word].append(doc_id)

    self.inverted_index = inverted_index

# Ne pas modifier les lignes ci-dessous
IRSystem.index = index
ir_system.index()
run_tests(ir_system, part=0)

Indexing...
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 2. - Recherche booléenne

Ce deuxième exercice a pour objectif d'implémenter la recherche booléenne. La requête `query` est une liste de mots _racinisés_ (stemming de Porter) et l'algorithme doit rechercher les documents qui contiennent TOUS ces mots.


Exercice : modifier la fonction `boolean_retrieve` pour implémenter la recherche booléenne.


Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [7]:
# Exercice 2. Recherche booléenne
def boolean_retrieve(self, query):
    """
    A partir d'une requête sous la forme d'une liste de mots *racinisés*,
    retourne la liste des documents dans lesquels *tous* ces mots apparaissent (ie une requête AND).
    Retourne une liste vide si la requête ne retourne aucun document.

    Dans le code ci-dessous, tous les documents répondent.
    """
    # Une requête vide ne retourne aucun document.
    if not query:
        return []

    # Récupère les documents contenant le premier terme.
    docs = set(self.inverted_index.get(query[0], []))

    # Conserve uniquement les documents contenant les autres termes.
    for word in query[1:]:
        docs.intersection_update(self.inverted_index.get(word, []))

        # Arrête la recherche si aucun document ne correspond.
        if not docs:
            break

    # Retourne les identifiants des documents dans l'ordre croissant.
    return sorted(docs)


# Ne pas modifier les lignes ci-dessous
IRSystem.boolean_retrieve = boolean_retrieve
run_tests(ir_system, part=1)


===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 3. - Calcul des poids TF-IDF des termes dans les documents

Dans ce troisième exercice, l'objectif est de pré-calculer les poids TF-IDF pour chaque terme dans chaque document. Pour ce faire, appliquer la formule vue en cours. Utiliser le logarithme en base 10.


Exercice : modifier la fonction `compute_tfidf` pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [8]:
import math

# Exercice 3. Calcul des scores TF-IDF
def compute_tfidf(self):
    """
    Calcule les poids TF-IDF des termes dans chaque document.

    Le TF est pondéré de manière logarithmique afin de réduire
    l'influence des termes très fréquents dans un document.

    L'IDF mesure la rareté d'un terme dans le corpus.
    Les résultats sont stockés dans self.tfidf.
    """
    print("Calculating tf-idf...")

    self.tfidf = defaultdict(Counter)
    N = len(self.docs)

    # Parcourt chaque terme du vocabulaire.
    for word in self.vocab:

        # Calcule le nombre de documents contenant le terme.
        df = len(self.inverted_index[word])

        # Évite une division par zéro.
        if df == 0:
            continue

        # Calcule l'IDF en base 10.
        idf = math.log10(N / df)

        # Calcule le TF-IDF dans les documents contenant le terme.
        for doc_id in self.inverted_index[word]:

            # Récupère le nombre d'occurrences du terme.
            tf = self.tf[doc_id][word]

            # Applique la pondération logarithmique du TF.
            tf_log = 1 + math.log10(tf)

            # Calcule le poids TF-IDF.
            self.tfidf[doc_id][word] = tf_log * idf


# Ne pas modifier les lignes ci-dessous
IRSystem.compute_tfidf = compute_tfidf
ir_system.compute_tfidf()
run_tests(ir_system, part=2)



Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


#### Question - Du bruit dans le corpus ?

Les livres proviennent du [Projet Gutenberg](https://www.gutenberg.org/). Ouvrez quelques fichiers du dossier `data/RiderHaggard/raw` et regardez leur dernière ligne : la plupart se terminent par une mention ajoutée par le Projet Gutenberg, du type « End of the Project Gutenberg EBook of She, by H. Rider Haggard ».

1. Affichez, pour les termes `gutenberg`, `ebook` et `project`, le nombre de documents qui les contiennent (df) et leur idf. Rappel : `self.inverted_index` attend des mots racinisés, utilisez `get_posting_unstemmed`.
2. Pourquoi le poids TF-IDF de `gutenberg` reste-t-il faible dans tous les documents, alors que ce mot n'a rien à voir avec les romans ?
3. Quel serait l'idf de `gutenberg` si un seul livre contenait cette mention ? Quel risque cela poserait-il pour la recherche ?
4. Quel traitement proposeriez-vous avant l'indexation ? Faites le lien avec la partie du cours « Représenter les documents ».

In [9]:


# Question - Du bruit dans le corpus ?
N = len(ir_system.docs)

for word in ['gutenberg', 'ebook', 'project']:

    # Récupère les documents contenant le terme après racinisation.
    posting = ir_system.get_posting_unstemmed(word)

    # Calcule la fréquence documentaire et l'IDF.
    df = len(posting)
    idf = math.log10(N / df) if df > 0 else None

    print(f"{word} : df = {df}, idf = {idf}")


# Vos réponses aux questions 2 à 4 :

# 2. Le terme "gutenberg" apparaît dans 52 documents sur 60.
# Son IDF est très faible (0,0621), ce qui limite son poids TF-IDF.
# Il provient des mentions du Projet Gutenberg plutôt que des romans.

# 3. Si un seul document contenait "gutenberg", son IDF serait
# log10(60) ≈ 1,7782. Une mention technique pourrait alors recevoir
# un poids élevé et fausser le classement des documents.

# 4. Il faudrait supprimer les en-têtes, pieds de page et mentions
# du Projet Gutenberg avant l'indexation.
# Ce prétraitement réduirait le bruit et améliorerait la représentation
# du contenu réel des documents.

gutenberg : df = 52, idf = 0.062147906748844434
ebook : df = 38, idf = 0.19836765376683346
project : df = 60, idf = 0.0


### Exercice 4. - Calcul de la similarité cosinus.

Dans ce quatrième exercice, l'objectif est de calculer la similarité cosinus entre le vecteur requête `query` et chaque vecteur document. Pour ce faire, appliquer la formule vue en cours en considérant les éléments suivants :
- **Vecteurs documents** : poids TF-IDF calculés à l'exercice 3, **normalisés L2** (on divise par la norme du vecteur document).
- **Vecteur requête** : poids TF logarithmique `1 + log10(tf)`, **sans IDF et sans normalisation**.
- Utiliser le logarithme en base 10.

Le score attendu est donc :

$$score(q, d) = \frac{\sum_{t \in q} w_{t,q} \cdot w_{t,d}}{\lVert d \rVert_2}$$

Ne pas normaliser le vecteur requête ne change pas le classement (la norme de la requête est la même pour tous les documents), mais change la valeur du score, qui est vérifiée par le test. En notation SMART, il s'agit du schéma **ltc.lnn**.

Exercice : modifier la fonction `rank_retrieve` pour implémenter la similarité cosinus.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [10]:
# Exercice 4. Similarité cosinus
def rank_retrieve(self, query):
    """
    À partir d'une requête (liste de mots), retourne les 10 meilleurs
    documents (docID, score), classés par score décroissant.

    Les documents utilisent les poids TF-IDF normalisés par leur norme L2.
    La requête utilise une pondération TF logarithmique,
    sans IDF et sans normalisation.
    """

    scores = [0.0 for _ in range(len(self.docs))]

    # Calcule les poids TF logarithmiques de la requête.
    query_tf = Counter(query)
    query_weights = {}

    for word, tf in query_tf.items():
        query_weights[word] = 1 + math.log10(tf)

    for d in range(len(self.docs)):

        # Récupère les poids TF-IDF du document.
        doc_weights = self.tfidf[d]

        # Calcule la norme L2 du document.
        norm = math.sqrt(sum(weight ** 2 for weight in doc_weights.values()))

        # Calcule le produit scalaire entre la requête et le document.
        produit_scalaire = 0.0
        for word, query_weight in query_weights.items():
            produit_scalaire += query_weight * doc_weights.get(word, 0.0)

        # Calcule la similarité en normalisant uniquement le document.
        scores[d] = produit_scalaire / norm if norm != 0 else 0.0

    # Tri des scores (conservé du code du professeur).
    ranking = [idx for idx, sim in sorted(enumerate(scores),
                                        key=lambda pair: pair[1], reverse=True)]
    results = []
    for i in range(10):
        results.append((ranking[i], scores[ranking[i]]))
    return results


# Ne pas modifier les lignes ci-dessous
IRSystem.rank_retrieve = rank_retrieve
run_tests(ir_system, part=3)

===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
